In [ ]:
!pip install -U transformers shap

from google.colab import drive
drive.mount('/content/drive')

LOAD_DIR = "/content/drive/MyDrive/vkr_model_bert"  # твоя папка с моделью

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(LOAD_DIR)
model = AutoModelForSequenceClassification.from_pretrained(LOAD_DIR).to(device)
model.eval()

print("Модель и токенайзер загружены.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Модель и токенайзер загружены.


In [ ]:
import shap
import numpy as np
import pandas as pd
import torch
import random  # Импортируем random для случайной выборки отзывов
import plotly.express as px
import os
import re
from collections import defaultdict
def predict_batch(texts, max_length=256, batch_size=16):
    """
    Возвращает предсказанные метки и вероятности для списка текстов.

    Классы:
    0 = Нейтральные
    1 = Негативные
    2 = Позитивные

    Аргументы:
    texts - список текстов для обработки
    max_length - максимальная длина текста
    batch_size - размер батча для обработки (по умолчанию 16)
    """
    if isinstance(texts, str):
        texts = [texts]
    texts = [str(t) for t in texts]

    all_preds = []
    all_probs = []

    # Обрабатываем тексты пакетами
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**enc)
            batch_probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
            batch_preds = np.argmax(batch_probs, axis=1)

        all_preds.extend(batch_preds)
        all_probs.extend(batch_probs)

    return np.array(all_preds), np.array(all_probs)

In [ ]:
# Функция для предсказания вероятности позитивного отзыва (для SHAP)
def predict_pos(texts, max_length=256):
    """
    Возвращает P(class=2) — вероятность ПОЛОЖИТЕЛЬНОГО отзыва.
    Именно этот класс мы анализируем через SHAP, чтобы найти плюсы и минусы.
    """
    if isinstance(texts, str):
        texts = [texts]

    processed = []
    for t in texts:
        if isinstance(t, list):
            processed.append(" ".join(t))
        else:
            processed.append(str(t))

    enc = tokenizer(
        processed,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
        # Индекс 2 — это положительный класс в данной модели
        pos_probs = probs[:, 2]
    return pos_probs

# Создаем explainer для SHAP анализ (ЭТО КРИТИЧЕСКИ ВАЖНО!)
print("Создаем SHAP explainer...")
masker = shap.maskers.Text(tokenizer=r"\W+")
explainer = shap.Explainer(predict_pos, masker=masker)
print("SHAP explainer успешно создан!")

Создаем SHAP explainer...
SHAP explainer успешно создан!


In [ ]:
# Расширяем список стоп-слов, чтобы убрать мусор из инсайтов
EXTRA_STOPWORDS = {
    "и", "в", "во", "на", "но", "что", "как", "к", "ко", "от", "до", "за", "из", "у",
    "по", "а", "не", "это", "так", "же", "то", "с", "со", "очень", "фен", "товар",
    "купить", "очень", "приобрести", "покупка", "товар", "вещь", "штука", "средство",
    "пользуюсь", "пользуюсь", "пользуюсь", "использую", "использую", "использую",
    "очень", "очень", "очень", "очень", "очень", "очень", "очень", "очень", "очень",
    "очень", "очень", "очень", "очень", "очень", "очень", "очень", "очень", "очень"
}

def pros_cons_from_reviews(texts, n_samples=50, top_k=5, min_shap_tokens_per_review=5):
    """
    texts — список текстов отзывов
    n_samples — количество отзывов для анализа SHAP
    top_k — сколько лучших фраз-инсайтов вернуть
    min_shap_tokens_per_review — минимальное количество токенов в отзыве для анализа
    """
    # 1. Жесткая фильтрация входящих текстов (защита от ошибок SHAP)
    valid_texts = []
    for t in texts:
        s_text = str(t).strip()
        if len(s_text) > 20:  # Игнорируем слишком короткие отзывы
            valid_texts.append(s_text)

    if not valid_texts:
        return ["Нет данных"], ["Нет данных"]

    # Подвыборка
    if len(valid_texts) > n_samples:
        texts_for_shap = random.sample(valid_texts, n_samples)
    else:
        texts_for_shap = valid_texts

    # 2. Считаем SHAP-значения
    shap_values = explainer(texts_for_shap)

    # 3. Извлекаем инсайты как ФРАЗЫ
    # Мы оцениваем влияние всего отзыва и выбираем самые характерные
    scored_sentences = []

    for i, ex in enumerate(shap_values):
        # Вес всего отзыва (сумма вкладов всех слов)
        total_impact = np.sum(ex.values)

        # Очищаем текст от лишних переносов строк для красоты
        clean_text = texts_for_shap[i].replace('\n', ' ').strip()

        scored_sentences.append((clean_text, total_impact))

    # Плюсы — отзывы с самым высоким положительным влиянием
    pros_raw = sorted(scored_sentences, key=lambda x: x[1], reverse=True)
    # Минусы — отзывы с самым сильным негативным влиянием (наименьшие значения)
    cons_raw = sorted(scored_sentences, key=lambda x: x[1], reverse=False)

    # Убираем дубликаты и короткие отрывки, оставляем топ
    pros = list(dict.fromkeys([p[0] for p in pros_raw if p[1] > 0.1]))[:top_k]
    cons = list(dict.fromkeys([c[0] for c in cons_raw if c[1] < 0]))[:top_k]

    return pros, cons

In [ ]:
# Функция для построения промта для LLM
def build_llm_prompt(product_name, product_category, pros, cons, tone="дружелюбный, но профессиональный"):
    """
    Генерирует промт для языковой модели на основе анализа отзывов.

    Аргументы:
    product_name - название товара
    product_category - категория товара
    pros - список ключевых преимуществ
    cons - список ключевых недостатков
    tone - желаемый тон сообщения

    Возвращает строку с промтом для LLM.
    """
    prompt = f"""
Ты — опытный маркетолог и копирайтер. Товар: "{product_name}", категория: {product_category}.

На основе анализа реальных отзывов клиентов известны следующие особенности.

Сильные стороны (прямые цитаты и факты):
"""

    for i, pro in enumerate(pros, 1):
        prompt += f"- {pro}\n"

    prompt += "\nСлабые стороны (реальные жалобы клиентов):\n"
    for i, con in enumerate(cons, 1):
        prompt += f"- {con}\n"

    prompt += f"""
Твоя задача:
1. Напиши 3 разных варианта рекламного текста для этого товара, подходящие для социальных сетей (Instagram, VK).
2. Тон: {tone}.
3. Не преувеличивай и не обещай того, что мы не контролируем
   (например, если клиенты жалуются на шум, не пиши "абсолютно бесшумный").
4. Добавь по одному короткому слогану (до 6–7 слов) к каждому варианту.

Ответ выдай в структурированном виде:
Вариант 1:
Текст: ...
Слоган: ...

Вариант 2:
...
"""
    return prompt

In [ ]:
# Загрузка данных с разделителем ";"
sushi_path = "/content/0008b828-f533-482b-b1e1-5fe24725c179.csv"  # ваш файл с несколькими товарами
df_sushi = pd.read_csv(sushi_path, sep=';')

# Проверка структуры данных
print("Первые 5 строк данных:")
print(df_sushi.head())
print("\nКолонки в датасете:", df_sushi.columns)
print(f"Количество уникальных товаров: {df_sushi['nmId'].nunique()}")

# Проверка наличия необходимых колонок
required_columns = ['nmId', 'rating', 'text']
if not all(col in df_sushi.columns for col in required_columns):
    raise ValueError(f"Файл должен содержать колонки: {required_columns}")

Первые 5 строк данных:
        nmId  rating                                               text
0  247407594       5        Хорошее полотенце. Брали на подарок тренеру
1  247407594       5     Полотенце супер! Хорошо впитывает! Компактное!
2  247407594       5  Класс, реально быстро сохнет. Идеально для дал...
3  247407594       5  Полотенце хорошее, выбрала цвет другой Удобно ...
4  247407594       5  Супер, не ожидал что так понравится, еще закаж...

Колонки в датасете: Index(['nmId', 'rating', 'text'], dtype='object')
Количество уникальных товаров: 3


In [ ]:
import random
# Получаем список уникальных артикулов
unique_nmIds = df_sushi['nmId'].unique()
print(f"\nНайдено {len(unique_nmIds)} уникальных товаров для анализа")

# Словарь для хранения результатов по каждому товару
all_results = {}

# Для каждого товара выполняем отдельный анализ
for nmId in unique_nmIds:
    print(f"\n{'='*60}")
    print(f"АНАЛИЗ ТОВАРА С АРТИКУЛОМ: {nmId}")
    print(f"{'='*60}")

    # Фильтруем отзывы только для текущего товара
    df_product = df_sushi[df_sushi['nmId'] == nmId].copy()
    print(f"Количество отзывов для анализа: {len(df_product)}")

    # Применяем модель к текстам отзывов с использованием батчей
    preds, probs = predict_batch(df_product["text"].tolist(), batch_size=8)

    # Добавляем результаты предсказаний в DataFrame
    df_product["pred_label"] = preds
    df_product["prob_neg"] = probs[:, 1]
    df_product["prob_pos"] = probs[:, 2]

    # Анализируем плюсы и минусы
    texts_product = df_product["text"].tolist()
    pros, cons = pros_cons_from_reviews(texts_product, n_samples=min(50, len(texts_product)), top_k=5)

    # Сохраняем результаты
    all_results[nmId] = {
        'dataframe': df_product,
        'pros': pros,
        'cons': cons,
        'stats': {
            'total_reviews': len(df_product),
            'positive': len(df_product[df_product['pred_label'] == 2]),
            'negative': len(df_product[df_product['pred_label'] == 1]),
            'neutral': len(df_product[df_product['pred_label'] == 0])
        }
    }

    # Выводим ключевые плюсы и минусы
    print(f"\n⭐ КЛЮЧЕВЫЕ ПЛЮСЫ (Мнение пользователей):")
    for i, p in enumerate(pros, 1):
        print(f"{i}. {p}")

    print(f"\n⚡ КЛЮЧЕВЫЕ МИНУСЫ (Жалобы):")
    for i, c in enumerate(cons, 1):
        print(f"{i}. {c}")

    # Создаем промт для LLM
    product_name = f"Товар {nmId}"
    product_category = "Техника для красоты"  # Можно улучшить, получая из данных
    final_prompt = build_llm_prompt(product_name, product_category, pros, cons)

    # Сохраняем промт
    all_results[nmId]['prompt'] = final_prompt

    # Сохраняем промт в файл
    with open(f"prompt_product_{nmId}.txt", "w", encoding="utf-8") as f:
        f.write(final_prompt)

    print(f"\nПромт для LLM сохранен в файл prompt_product_{nmId}.txt")


Найдено 3 уникальных товаров для анализа

АНАЛИЗ ТОВАРА С АРТИКУЛОМ: 247407594
Количество отзывов для анализа: 527


PartitionExplainer explainer: 51it [00:39,  1.02it/s]                        



⭐ КЛЮЧЕВЫЕ ПЛЮСЫ (Мнение пользователей):
1. Полотенце отличное, беру уже третье. Брала в подарок и теперь купила всей семье на море, и бассейн. Удобно что влажное полотенце убрал в чехол, или если на пляже в песке запачкалось и готово! Очень компактно в пляжной и дорожной сумке.
2. Отличное полотенце, особенно удобно брать в бассейн/аквапарк. Компактное. Хорошо впитывает и быстро сохнет. Купила всей семье и друзьям подарила.
3. Хорошее полотенце, отлично впитывает, очень быстро сохнет и очень компактно, удобно брать  в дорогу. Брала в подарок который очень понравился.
4. Красивое, лёгкое, компактное полотенце. Цвет яркий. Доставили очень быстро 👍 Все супер 😍 Спасибо большое за качественный товар. Брали на подарок, доставили все вовремя,  очень быстро!
5. Отлично, брала на подарок. Очень компактно для спортзала.

⚡ КЛЮЧЕВЫЕ МИНУСЫ (Жалобы):
1. Сыпется … отвратительно
2. Полотенце все в пятнах!!! Сразу не проверила, пришлось выбросить((((((

Промт для LLM сохранен в файл prompt_product_

PartitionExplainer explainer: 51it [00:26,  1.04it/s]



⭐ КЛЮЧЕВЫЕ ПЛЮСЫ (Мнение пользователей):
1. Хороший коврик, приятное матовое покрытие, мягкий, выглядит стильно. Немного скользит, но я использую его для занятий йогой в гамаке, возможно особенность тренировки, немного двигается по полу, не критично. Отличное качество!
2. Отличный коврик, не скользит приятный на ощупь. Комплект полный, есть чехол и резинка 👍🏻
3. Коврик отличного качества, прекрасно упакован. Надеюсь, что заниматься будет удобно.
4. Отличный коврик, красивый, не скользит, прочный. Очень удобно, что есть фиксатор и чехол. Пользуюсь уже больше полугода - все нравится, рекомендую
5. Приятный коврик, не скользит. Нравится цвет. Приехал весь комплект.

⚡ КЛЮЧЕВЫЕ МИНУСЫ (Жалобы):
1. Качество хорошее на первый взгляд Это не фуксия (яркая сторона), как заявлено было продавцом, а просто  розовый цвет, на карточке товара цвет не такой как в реальности. Неприятно, тк это был принципиальный момент, коврик покупался девушке в подарок с расчетом именно на этот цвет.
2. Из плюсов: в

PartitionExplainer explainer: 51it [00:29,  1.21it/s]                        



⭐ КЛЮЧЕВЫЕ ПЛЮСЫ (Мнение пользователей):
1. Супер ролл, обалденный массажный эффект, в меру мягко-твердый 🔥
2. Валик прекрасного качества. Упаковка на высшем уровне! Заказ упаковывала Маришка, спасибо ей большое! 🫂
3. Отличный валик! Все пришло хорошо упаковано.
4. Классный ролл, удобный, красивый! Цвет материал, все очень понравилось! Спасибо за качественный товар и подарки!
5. очень красивый цвет и удобный диаметр, до этого пользовалась более широким и коротким, этот размер удобнее!!  первый раз купила жесткий, мне очень нравится эффект! прям массаж ))

⚡ КЛЮЧЕВЫЕ МИНУСЫ (Жалобы):
1. К товару вопросов нет. Но я выбрала другой. В заказе было два валика. При этом деньги списали за оба. И статус их как доставленные. Как вернуть деньги за выкупленный товар? Служба поддержки wb бесполезна.
2. Пишут 3 в 1 а пришло 2 в 1 Нет гантельки
3. Рпзмер, упаковка Слишком жесткий, на спиге от использования появились синяки
4. Очень жёсткий не смогла его применять
5. В наличии мячик и валик, ролика н

In [ ]:
# Создаем папку для сохранения результатов
os.makedirs("product_analysis", exist_ok=True)
print("\nСоздана папка для сохранения результатов: product_analysis")

# Генерация визуализации для каждого товара
for nmId, results in all_results.items():
    df_product = results['dataframe']

    # Подготовка данных для Pie Chart
    counts = df_product['pred_label'].value_counts().sort_index()
    pie_data = pd.DataFrame({
        'Sentiment': ['Нейтрально', 'Негативно', 'Позитивно'],
        'Count': [counts.get(0, 0), counts.get(1, 0), counts.get(2, 0)],
        'Color': ['#808080', '#FF0000', '#008000']
    })

    # Создаем Pie Chart
    fig_pie = px.pie(
        pie_data,
        values='Count',
        names='Sentiment',
        title=f'Общий баланс тональности отзывов для товара {nmId}',
        color='Sentiment',
        color_discrete_map={row['Sentiment']: row['Color'] for _, row in pie_data.iterrows()},
        hole=0.4
    )
    fig_pie.update_traces(textposition='inside', textinfo='percent+label')

    # Сохраняем график
    fig_pie.write_html(f"product_analysis/sentiment_analysis_{nmId}.html")

    # Сохраняем статистику
    stats = results['stats']
    with open(f"product_analysis/stats_{nmId}.txt", "w", encoding="utf-8") as f:
        f.write(f"Статистика для товара {nmId}:\n")
        f.write(f"Всего отзывов: {stats['total_reviews']}\n")
        f.write(f"Позитивных: {stats['positive']} ({stats['positive']/stats['total_reviews']:.1%})\n")
        f.write(f"Негативных: {stats['negative']} ({stats['negative']/stats['total_reviews']:.1%})\n")
        f.write(f"Нейтральных: {stats['neutral']} ({stats['neutral']/stats['total_reviews']:.1%})\n\n")

        f.write("КЛЮЧЕВЫЕ ПЛЮСЫ:\n")
        for i, p in enumerate(results['pros'], 1):
            f.write(f"{i}. {p}\n")

        f.write("\nКЛЮЧЕВЫЕ МИНУСЫ:\n")
        for i, c in enumerate(results['cons'], 1):
            f.write(f"{i}. {c}\n")

# Создание сводного отчета по всем товарам
with open("product_analysis/summary_report.txt", "w", encoding="utf-8") as f:
    f.write("СВОДНЫЙ ОТЧЕТ ПО ВСЕМ ТОВАРАМ\n")
    f.write("="*50 + "\n\n")

    for nmId, results in all_results.items():
        stats = results['stats']
        f.write(f"Товар {nmId} (Всего отзывов: {stats['total_reviews']})\n")
        f.write(f"- Позитивных: {stats['positive']} ({stats['positive']/stats['total_reviews']:.1%})\n")
        f.write(f"- Негативных: {stats['negative']} ({stats['negative']/stats['total_reviews']:.1%})\n")
        f.write(f"- Нейтральных: {stats['neutral']} ({stats['neutral']/stats['total_reviews']:.1%})\n\n")

        f.write("ТОП-3 ПЛЮСА:\n")
        for i, p in enumerate(results['pros'][:3], 1):
            f.write(f"{i}. {p[:100]}{'...' if len(p) > 100 else ''}\n")
        f.write("\n")

        f.write("ТОП-3 МИНУСА:\n")
        for i, c in enumerate(results['cons'][:3], 1):
            f.write(f"{i}. {c[:100]}{'...' if len(c) > 100 else ''}\n")
        f.write("\n" + "-"*50 + "\n\n")

print("\nАнализ завершен. Все результаты сохранены в папке product_analysis/")


Создана папка для сохранения результатов: product_analysis

Анализ завершен. Все результаты сохранены в папке product_analysis/


In [ ]:
# Дополнительно: показать пример одного из сгенерированных промтов
if unique_nmIds.size > 0:
    sample_nmId = unique_nmIds[0]
    print(f"\n{'='*60}")
    print(f"ПРИМЕР ПРОМТА ДЛЯ ТОВАРА {sample_nmId}")
    print(f"{'='*60}")
    print(all_results[sample_nmId]['prompt'])
else:
    print("\nНет данных для отображения примера промта.")


ПРИМЕР ПРОМТА ДЛЯ ТОВАРА 247407594

Ты — опытный маркетолог и копирайтер. Товар: "Товар 247407594", категория: Техника для красоты.

На основе анализа реальных отзывов клиентов известны следующие особенности.

Сильные стороны (прямые цитаты и факты):
- Полотенце отличное, беру уже третье. Брала в подарок и теперь купила всей семье на море, и бассейн. Удобно что влажное полотенце убрал в чехол, или если на пляже в песке запачкалось и готово! Очень компактно в пляжной и дорожной сумке.
- Отличное полотенце, особенно удобно брать в бассейн/аквапарк. Компактное. Хорошо впитывает и быстро сохнет. Купила всей семье и друзьям подарила.
- Хорошее полотенце, отлично впитывает, очень быстро сохнет и очень компактно, удобно брать  в дорогу. Брала в подарок который очень понравился.
- Красивое, лёгкое, компактное полотенце. Цвет яркий. Доставили очень быстро 👍 Все супер 😍 Спасибо большое за качественный товар. Брали на подарок, доставили все вовремя,  очень быстро!
- Отлично, брала на подарок. Оч

In [ ]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_batch

# Твои данные (проверь еще раз!)
DB_HOST = "rc1a-a7qgfftqd5ql9000.mdb.yandexcloud.net"
DB_NAME = "vkr_db"
DB_USER = "user"
DB_PASS = "user1234"

try:
    # 1. Загружаем CSV
    df = pd.read_csv('e311401e-0ccf-4e93-a1e3-b2158a9d6e5f.csv', sep=';')
    # Берем только нужные колонки и убираем пустые строки, если они есть
    data_to_insert = df[['nmId', 'text']].dropna().values.tolist()

    # 2. Подключаемся
    conn = psycopg2.connect(
        host=DB_HOST,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
        port=6432,
        sslmode='require'
    )
    cursor = conn.cursor()

    # 3. Массовая вставка (это займет 2-3 секунды вместо 5 минут)
    print(f"🚀 Начинаю загрузку {len(data_to_insert)} строк...")
    query = "INSERT INTO reviews (nm_id, review_text) VALUES (%s, %s)"
    execute_batch(cursor, query, data_to_insert)

    # ОБЯЗАТЕЛЬНО сохраняем изменения
    conn.commit()

    print(f"✅ Успех! В базу залито {len(data_to_insert)} строк.")

except Exception as e:
    print(f"❌ Произошла ошибка: {e}")

finally:
    if 'conn' in locals() and conn:
        cursor.close()
        conn.close()

❌ Произошла ошибка: [Errno 2] No such file or directory: 'e311401e-аукауца0ccf-4e93-a1e3-b2158a9d6e5f.csv'


In [ ]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_batch
import os

# Параметры подключения к БД
DB_HOST = "rc1a-a7qgfftqd5ql9000.mdb.yandexcloud.net"
DB_NAME = "vkr_db"
DB_USER = "user"
DB_PASS = "user1234"

# Путь к папке с результатами анализа
ANALYSIS_DIR = "product_analysis"

try:
    # Подключаемся к базе данных
    conn = psycopg2.connect(
        host=DB_HOST,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
        port=6432,
        sslmode='require'
    )
    cursor = conn.cursor()
    print("✅ Подключение к базе данных установлено")

    # 1. Проверяем структуру таблицы product_summary и добавляем колонку для графиков при необходимости
    cursor.execute("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_name = 'product_summary' AND column_name = 'chart_html'
    """)

    if not cursor.fetchone():
        print("🔧 Добавляем колонку chart_html в таблицу product_summary")
        cursor.execute("ALTER TABLE product_summary ADD COLUMN chart_html TEXT")
        conn.commit()

    # 2. Обновляем таблицу reviews, заполняя sentiment и confidence
    print("\n🔄 Начинаем обновление таблицы reviews...")

    # Получаем список файлов результатов
    stats_files = [f for f in os.listdir(ANALYSIS_DIR) if f.startswith('stats_') and f.endswith('.txt')]

    total_reviews_updated = 0
    for stats_file in stats_files:
        # Извлекаем nmId из имени файла
        nmId = stats_file.replace('stats_', '').replace('.txt', '')

        # Пропускаем артикул 287657449, так как данные для него уже есть
        if nmId == "287657449":
            print(f"⏩ Пропускаем артикул {nmId} (данные уже в базе)")
            continue

        print(f"📝 Обрабатываем артикул {nmId}")

        # Читаем данные из stats-файла
        with open(os.path.join(ANALYSIS_DIR, stats_file), 'r', encoding='utf-8') as f:
            stats_content = f.read()

        # Извлекаем статистику
        total_reviews = int(re.search(r'Всего отзывов: (\d+)', stats_content).group(1))
        positive = int(re.search(r'Позитивных: (\d+)', stats_content).group(1))
        negative = int(re.search(r'Негативных: (\d+)', stats_content).group(1))
        neutral = int(re.search(r'Нейтральных: (\d+)', stats_content).group(1))

        print(f"   Найдено отзывов: {total_reviews} (позитивных: {positive}, негативных: {negative}, нейтральных: {neutral})")

        # Загружаем исходные данные для этого артикула
        df_sushi = pd.read_csv('0008b828-f533-482b-b1e1-5fe24725c179.csv', sep=';')
        df_product = df_sushi[df_sushi['nmId'] == int(nmId)].copy() if nmId.isdigit() else df_sushi[df_sushi['nmId'] == nmId].copy()

        # Применяем модель к текстам отзывов
        preds, probs = predict_batch(df_product["text"].tolist())

        # Добавляем результаты предсказаний в DataFrame
        df_product["pred_label"] = preds
        if len(probs.shape) == 1:
            # Случай 1: Одномерный массив (бинарная классификация)
            df_product["prob_pos"] = probs
            df_product["prob_neg"] = 1 - probs
            df_product["prob_neutral"] = 0  # Нейтральный класс отсутствует
        else:
            # Случай 2: Двумерный массив (многоклассовая классификация)
            df_product["prob_neg"] = probs[:, 1]
            df_product["prob_pos"] = probs[:, 2]
            df_product["prob_neutral"] = probs[:, 0]

        # Подготавливаем данные для обновления
        update_data = []
        for _, row in df_product.iterrows():
            # Вычисляем confidence как вероятность предсказанного класса
            if row['pred_label'] == 0:  # нейтральный
                confidence = 1 - row['prob_neg'] - row['prob_pos']
            elif row['pred_label'] == 1:  # негативный
                confidence = row['prob_neg']
            else:  # позитивный
                confidence = row['prob_pos']

            update_data.append((
                int(row['pred_label']),
                float(confidence),
                nmId,
                str(row['text'])
            ))

        # Массовое обновление
        if update_data:
            update_query = """
                UPDATE reviews
                SET sentiment = %s, confidence = %s
                WHERE nm_id = %s AND review_text = %s
            """
            execute_batch(cursor, update_query, update_data)
            conn.commit()
            total_reviews_updated += len(update_data)
            print(f"   ✅ Обновлено {len(update_data)} отзывов для артикула {nmId}")

    print(f"\n✅ Таблица reviews обновлена. Всего обновлено отзывов: {total_reviews_updated}")

    # 3. Обновляем таблицу product_summary с новыми данными и графиками
    print("\n🔄 Начинаем обновление таблицы product_summary...")

    for stats_file in stats_files:
        nmId = stats_file.replace('stats_', '').replace('.txt', '')

        # Пропускаем артикул 287657449
        if nmId == "287657449":
            print(f"⏩ Пропускаем артикул {nmId} в таблице product_summary (данные уже в базе)")
            continue

        print(f"📝 Обрабатываем артикул {nmId} для таблицы product_summary")

        # Читаем текстовый отчет
        with open(os.path.join(ANALYSIS_DIR, stats_file), 'r', encoding='utf-8') as f:
            summary_text = f.read()

        # Читаем HTML график
        chart_file = f"sentiment_analysis_{nmId}.html"
        chart_html = ""
        if os.path.exists(os.path.join(ANALYSIS_DIR, chart_file)):
            with open(os.path.join(ANALYSIS_DIR, chart_file), 'r', encoding='utf-8') as f:
                chart_html = f.read()

        # Проверяем существование записи
        cursor.execute("SELECT COUNT(*) FROM product_summary WHERE nm_id = %s", (nmId,))
        exists = cursor.fetchone()[0] > 0

        if exists:
            # Обновляем существующую запись
            update_query = """
                UPDATE product_summary
                SET summary_text = %s, chart_html = %s
                WHERE nm_id = %s
            """
            cursor.execute(update_query, (summary_text, chart_html, nmId))
            print(f"   ✏️ Обновлена запись для артикула {nmId}")
        else:
            # Добавляем новую запись
            insert_query = """
                INSERT INTO product_summary (nm_id, summary_text, chart_html)
                VALUES (%s, %s, %s)
            """
            cursor.execute(insert_query, (nmId, summary_text, chart_html))
            print(f"   ➕ Добавлена новая запись для артикула {nmId}")

    conn.commit()
    print("✅ Таблица product_summary успешно обновлена")

    # 4. Проверка обновленных данных
    print("\n🔍 Проверка обновленных данных:")
    cursor.execute("""
        SELECT nm_id, COUNT(*)
        FROM reviews
        WHERE sentiment IS NOT NULL
        GROUP BY nm_id
    """)

    for row in cursor.fetchall():
        print(f"   Артикул {row[0]}: {row[1]} отзывов с проставленной тональностью")

    cursor.execute("""
        SELECT nm_id, LENGTH(chart_html)
        FROM product_summary
        WHERE chart_html IS NOT NULL
    """)

    for row in cursor.fetchall():
        print(f"   Артикул {row[0]}: график сохранен ({row[1]} символов)")

    print("\n🎉 Все данные успешно перенесены в базу данных!")

except Exception as e:
    print(f"❌ Произошла ошибка: {e}")
    import traceback
    traceback.print_exc()

finally:
    if 'conn' in locals() and conn:
        cursor.close()
        conn.close()
        print("\n🔌 Соединение с базой данных закрыто")

✅ Подключение к базе данных установлено

🔄 Начинаем обновление таблицы reviews...
📝 Обрабатываем артикул 165879171
   Найдено отзывов: 379 (позитивных: 308, негативных: 22, нейтральных: 49)
📝 Обрабатываем артикул 684264302
   Найдено отзывов: 772 (позитивных: 599, негативных: 49, нейтральных: 124)
📝 Обрабатываем артикул 175573791
   Найдено отзывов: 537 (позитивных: 363, негативных: 40, нейтральных: 134)
📝 Обрабатываем артикул 232017200
   Найдено отзывов: 540 (позитивных: 435, негативных: 38, нейтральных: 67)
   ✅ Обновлено 540 отзывов для артикула 232017200
📝 Обрабатываем артикул 174812760
   Найдено отзывов: 449 (позитивных: 349, негативных: 25, нейтральных: 75)
   ✅ Обновлено 449 отзывов для артикула 174812760
📝 Обрабатываем артикул 247407594
   Найдено отзывов: 527 (позитивных: 446, негативных: 8, нейтральных: 73)
   ✅ Обновлено 527 отзывов для артикула 247407594

✅ Таблица reviews обновлена. Всего обновлено отзывов: 1516

🔄 Начинаем обновление таблицы product_summary...
📝 Обрабат

In [ ]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_batch
import os

# Параметры подключения к БД
DB_HOST = "rc1a-a7qgfftqd5ql9000.mdb.yandexcloud.net"
DB_NAME = "vkr_db"
DB_USER = "user"
DB_PASS = "user1234"

# Путь к файлу с отзывами
REVIEWS_FILE = "0008b828-f533-482b-b1e1-5fe24725c179.csv"  # Ваш файл с отзывами

try:
    # Подключаемся к базе данных
    conn = psycopg2.connect(
        host=DB_HOST,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
        port=6432,
        sslmode='require'
    )
    cursor = conn.cursor()
    print("✅ Подключение к базе данных установлено")

    # Загружаем данные из файла с отзывами
    print(f"🔄 Загрузка данных из файла: {REVIEWS_FILE}")
    df_reviews = pd.read_csv(REVIEWS_FILE, sep=';')

    # Проверяем структуру данных
    if 'nmId' not in df_reviews.columns or 'text' not in df_reviews.columns:
        raise ValueError("Файл должен содержать колонки 'nmId' и 'text'")

    print(f"   Найдено {len(df_reviews)} отзывов для обработки")

    # Применяем модель BERT для анализа тональности
    print("🤖 Применение модели BERT для анализа тональности...")
    preds, probs = predict_batch(df_reviews["text"].tolist())

    # Добавляем результаты предсказаний в DataFrame
    df_reviews["pred_label"] = preds
    if len(probs.shape) == 1:
        # Случай 1: Одномерный массив (бинарная классификация)
        df_reviews["prob_pos"] = probs
        df_reviews["prob_neg"] = 1 - probs
        df_reviews["prob_neutral"] = 0  # Нейтральный класс отсутствует
    else:
        # Случай 2: Двумерный массив (многоклассовая классификация)
        df_reviews["prob_neg"] = probs[:, 1]
        df_reviews["prob_pos"] = probs[:, 2]
        df_reviews["prob_neutral"] = probs[:, 0]

    # Подготавливаем данные для вставки
    insert_data = []
    for _, row in df_reviews.iterrows():
        # Вычисляем confidence как вероятность предсказанного класса
        if row['pred_label'] == 0:  # нейтральный
            confidence = 1 - row['prob_neg'] - row['prob_pos']
        elif row['pred_label'] == 1:  # негативный
            confidence = row['prob_neg']
        else:  # позитивный
            confidence = row['prob_pos']

        insert_data.append((
            int(row['nmId']),  # nm_id
            str(row['text']),  # review_text
            int(row['pred_label']),  # sentiment
            float(confidence)  # confidence
        ))

    # Массовая вставка данных в таблицу reviews
    print(f"🚀 Начинаем загрузку {len(insert_data)} строк в таблицу reviews...")
    insert_query = """
        INSERT INTO reviews (nm_id, review_text, sentiment, confidence)
        VALUES (%s, %s, %s, %s)
    """
    execute_batch(cursor, insert_query, insert_data)
    conn.commit()

    print(f"✅ Успех! В базу залито {len(insert_data)} строк.")

except Exception as e:
    print(f"❌ Произошла ошибка: {e}")
    import traceback
    traceback.print_exc()

finally:
    if 'conn' in locals() and conn:
        cursor.close()
        conn.close()
        print("\n🔌 Соединение с базой данных закрыто")